In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
import os 
import sys
from collections import Counter
from imblearn.under_sampling import RandomUnderSampler

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
BASE_DIR = Path.cwd() 
print(BASE_DIR)

c:\Users\Neezy\Videos\attacks_improved_v2\Attacks-Improved_v2\Random_forest


In [4]:
project_root = os.path.abspath('..')
if project_root not in sys.path: 
    sys.path.append(project_root)

In [5]:
# import attacks
from attack_functionality.functionality_preserving_attack import FunctionalityPreservingAttack 
attack = FunctionalityPreservingAttack()

In [6]:
data = pd.read_csv(BASE_DIR.parent/'data/balanced_df.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [7]:
data['label'].value_counts()

label
0    570
1    570
Name: count, dtype: int64

In [8]:
data.head()

,Sample Name,category,SetErrorMode,OleInitialize,LdrGetDllHandle,LdrLoadDll,LdrGetProcedureAddress,NtOpenSection,NtMapViewOfSection,RegOpenKeyExW,RegQueryValueExW,RegCloseKey,NtClose,NtOpenKey,NtQueryValueKey,GetSystemWindowsDirectoryW,NtCreateFile,NtCreateSection,RegOpenKeyExA,CreateActCtxW,GetSystemDirectoryW,GetVolumeNameForVolumeMountPointW,NtDuplicateObject,LoadStringW,NtCreateMutant,GetNativeSystemInfo,RegEnumKeyW,NtQuerySystemInformation,RegQueryValueExA,NtQueryDirectoryFile,GlobalMemoryStatusEx,CoCreateInstance,NtAllocateVirtualMemory,CreateDirectoryW,DeleteFileW,GetFileSizeEx,NtReadFile,GetFileInformationByHandleEx,GetSystemTimeAsFileTime,GetVolumePathNamesForVolumeNameW,LdrUnloadDll,CoInitializeEx,NtOpenProcess,CoUninitialize,NtFreeVirtualMemory,NtOpenFile,NtQueryInformationFile,GetFileAttributesW,FindFirstFileExW,NtQueryAttributesFile,NtUnmapViewOfSection,SetFilePointerEx,SetFilePointer,GetTempPathW,GetFileSize,NtWriteFile,FindResourceExW,LoadResource,SHGetFolderPathW,NtProtectVirtualMemory,GetFileType,ReadProcessMemory,GetForegroundWindow,GetSystemMetrics,SetFileTime,NtSetInformationFile,SearchPathW,NtOpenMutant,RegEnumKeyExW,DrawTextExW,GetAsyncKeyState,GetDiskFreeSpaceExW,GetKeyState,FindWindowW,FindWindowExA,CreateThread,MoveFileWithProgressW,SetFileAttributesW,RemoveDirectoryW,NtTerminateProcess,CreateToolhelp32Snapshot,Process32FirstW,Process32NextW,FindWindowExW,SetEndOfFile,GetCursorPos,SetUnhandledExceptionFilter,OutputDebugStringA,GetSystemInfo,FindResourceW,SizeofResource,NtDelayExecution,GetKeyboardState,WSAStartup,socket,setsockopt,NtDeviceIoControlFile,closesocket,GetBestInterfaceEx,GetAdaptersAddresses,NtQueryKey,RegCreateKeyExW,GetAddrInfoW,GetUserNameExW,RegSetValueExW,RegDeleteValueW,InternetQueryOptionA,URLDownloadToFileW,IsDebuggerPresent,CreateProcessInternalW,GetTimeZoneInformation,LookupAccountSidW,SendNotifyMessageW,UuidCreate,GetFileVersionInfoSizeW,GetFileVersionInfoW,NtEnumerateValueKey,EnumWindows,OpenSCManagerW,GetComputerNameW,GetUserNameW,NetShareEnum,GetFileInformationByHandle,DeviceIoControl,ShellExecuteExW,RegQueryInfoKeyW,RegEnumValueW,RegDeleteKeyW,NtReadVirtualMemory,NtOpenKeyEx,NtSetValueKey,NtCreateKey,GetVolumePathNameW,GetFileAttributesExW,GetUserNameExA,RegCreateKeyExA,CryptAcquireContextW,NtEnumerateKey,NtDeleteKey,OpenServiceW,NtOpenDirectoryObject,CreateJobObjectW,SetInformationJobObject,RegEnumKeyExA,GetShortPathNameW,LoadStringA,FindResourceA,DrawTextExA,RegQueryInfoKeyA,RegSetValueExA,SHGetSpecialFolderLocation,NtCreateThreadEx,NtResumeThread,gethostbyname,GetSystemDirectoryA,FindResourceExA,GetDiskFreeSpaceW,CertOpenStore,CryptDecodeObjectEx,CertControlStore,CryptHashData,NtOpenThread,MessageBoxTimeoutW,LookupPrivilegeValueW,CryptAcquireContextA,SetFileInformationByHandle,RemoveDirectoryA,SetWindowsHookExW,CopyFileW,GetFileVersionInfoSizeExW,GetFileVersionInfoExW,CoInitializeSecurity,WSASocketW,WSAConnect,UnhookWindowsHookEx,CertOpenSystemStoreW,getaddrinfo,InternetCrackUrlW,CoCreateInstanceEx,CoGetClassObject,IWbemServices_ExecQuery,SetStdHandle,GlobalMemoryStatus,NetGetJoinInformation,CryptCreateHash,GetComputerNameA,InternetOpenA,InternetOpenUrlA,InternetCloseHandle,ReadCabinetState,InternetOpenW,InternetConnectW,HttpOpenRequestW,HttpSendRequestW,NtDeleteValueKey,HttpQueryInfoA,RegEnumValueA,CryptProtectMemory,CreateServiceW,WriteConsoleA,CopyFileA,WriteProcessMemory,SendNotifyMessageA,RegDeleteKeyA,WriteConsoleW,JsGlobalObjectDefaultEvalHelper,ObtainUserAgentString,StartServiceW,NtQueueApcThread,RtlAddVectoredContinueHandler,CryptExportKey,CryptGenKey,CryptEncrypt,NetUserGetInfo,GetUserNameA,InternetOpenUrlW,GetAdaptersInfo,Module32FirstW,NtGetContextThread,Module32NextW,RtlAddVectoredExceptionHandler,NtSuspendThread,OpenSCManagerA,OpenServiceA,NtQueryMultipleValueKey,MessageBoxTimeoutA,ControlService,NtTerminateThread,EncryptMessage,DecryptMessage,DeleteService,FindWindowA,RtlRemoveVectoredExceptionHandler,ioctlsocket,connect,select,SetWindowsHookExA,CreateServic

In [9]:
features = ['Sample Name', 'category', 'LdrGetProcedureAddress', 'GetSystemInfo', 'GetNativeSystemInfo', 'NtQuerySystemInformation',  'CreateToolhelp32Snapshot', 'Process32FirstW', 'RegOpenKeyExW', 'RegQueryValueExW',
            'RegCloseKey', 'NtCreateFile', 'GetDiskFreeSpaceW', 'CryptAcquireContextW', 'CryptGenKey', 
            'CryptExportKey', 'RegCreateKeyExW', 'RegSetValueExW', 'CreateThread', 'FindFirstFileExW',
            'NtWriteFile', 'ShellExecuteExW', 'InternetOpenW', 
            'InternetConnectW', 'HttpOpenRequestW', 'InternetReadFile', 'InternetOpenUrlA', 
            'InternetCloseHandle']

In [10]:
benign_features = ['LdrGetDllHandle',
 'RegOpenKeyExW',
 'RegQueryValueExW',
 'RegCloseKey',
 'GetSystemWindowsDirectoryW',
 'CreateActCtxW',
 'GetSystemDirectoryW',
 'GetVolumeNameForVolumeMountPointW',
 'RegEnumKeyW',
 'GlobalMemoryStatusEx',
 'CreateDirectoryW',
 'GetVolumePathNamesForVolumeNameW',
 'LdrUnloadDll',
 'GetFileAttributesW',
 'SHGetFolderPathW',
 'GetSystemMetrics',
 'SetFileTime',
 'RegEnumKeyExW',
 'DrawTextExW',
 'FindWindowW',
 'FindWindowExA',
 'RemoveDirectoryW',
 'GetKeyboardState',
 'SendNotifyMessageW',
 'GetFileVersionInfoSizeW',
 'EnumWindows',
 'GetUserNameExA',
 'RegCreateKeyExA',
 'SetInformationJobObject',
 'DrawTextExA',
 'RegSetValueExA',
 'SHGetSpecialFolderLocation',
 'gethostbyname',
 'CertOpenStore',
 'CertControlStore',
 'SetWindowsHookExW',
 'WSASocketW',
 'WSAConnect',
 'UnhookWindowsHookEx',
 'CertOpenSystemStoreW',
 'HttpQueryInfoA',
 'CryptProtectMemory',
 'WriteProcessMemory',
 'SendNotifyMessageA',
 'JsGlobalObjectDefaultEvalHelper',
 'ObtainUserAgentString',
 'label'
 ]

In [11]:
benign_filtered = [item for item in benign_features if item not in features]

In [12]:
len(benign_filtered)

44

In [13]:
functionality_features = features.extend(benign_filtered)

In [14]:
print(len(features))

72


In [15]:
df = data[features]

In [16]:
df.head()

,Sample Name,category,LdrGetProcedureAddress,GetSystemInfo,GetNativeSystemInfo,NtQuerySystemInformation,CreateToolhelp32Snapshot,Process32FirstW,RegOpenKeyExW,RegQueryValueExW,RegCloseKey,NtCreateFile,GetDiskFreeSpaceW,CryptAcquireContextW,CryptGenKey,CryptExportKey,RegCreateKeyExW,RegSetValueExW,CreateThread,FindFirstFileExW,NtWriteFile,ShellExecuteExW,InternetOpenW,InternetConnectW,HttpOpenRequestW,InternetReadFile,InternetOpenUrlA,InternetCloseHandle,LdrGetDllHandle,GetSystemWindowsDirectoryW,CreateActCtxW,GetSystemDirectoryW,GetVolumeNameForVolumeMountPointW,RegEnumKeyW,GlobalMemoryStatusEx,CreateDirectoryW,GetVolumePathNamesForVolumeNameW,LdrUnloadDll,GetFileAttributesW,SHGetFolderPathW,GetSystemMetrics,SetFileTime,RegEnumKeyExW,DrawTextExW,FindWindowW,FindWindowExA,RemoveDirectoryW,GetKeyboardState,SendNotifyMessageW,GetFileVersionInfoSizeW,EnumWindows,GetUserNameExA,RegCreateKeyExA,SetInformationJobObject,DrawTextExA,RegSetValueExA,SHGetSpecialFolderLocation,gethostbyname,CertOpenStore,CertControlStore,SetWindowsHookExW,WSASocketW,WSAConnect,UnhookWindowsHookEx,CertOpenSystemStoreW,HttpQueryInfoA,CryptProtectMemory,WriteProcessMemory,SendNotifyMessageA,JsGlobalObjectDefaultEvalHelper,ObtainUserAgentString,label
0,Database_Q-Eye Portable.exe,Development,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,Database_DB Browser for SQLite Portable.exe,Development,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Database_Database Browser Portable.exe,Development,322,0,1,1,1,1,714,2343,579,237,0,0,0,0,0,0,1,48,1517,0,0,0,0,0,0,0,209,7,1,16,4,111,1,122,8,9,318,2,1557,34,192,127,1,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,Action Games_AssaultCube Portable.exe,Games,150,0,1,3,0,0,709,2339,574,2023,0,0,0,0,0,0,1,74,9313,0,0,0,0,0,0,0,48,7,1,3,4,111,1,5900,8,11,9267,1,3087,1895,192,169,1,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Astronomy_Stellarium Portable.exe,Education,319,0,1,1,1,1,890,2407,752,3607,0,0,0,0,0,0,1,60,47827,0,0,0,0,0,0,0,195,7,2,16,4,111,1,1121,8,10,7970,2,12329,3413,358,294,1,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [17]:
X_bal = df.iloc[:, 2:-1].values
y_bal = df.iloc[:,-1].values
original_indices = np.arange(len(X_bal)) 
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X_bal, y_bal, original_indices, test_size=0.2, random_state=42)
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n {cm}") 
cr = classification_report(y_test, y_pred)
print(f"Classification report:\n {cr}")

Accuracy: 94.30%
Confusion Matrix:
 [[125   3]
 [ 10  90]]
Classification report:
               precision    recall  f1-score   support

           0       0.93      0.98      0.95       128
           1       0.97      0.90      0.93       100

    accuracy                           0.94       228
   macro avg       0.95      0.94      0.94       228
weighted avg       0.94      0.94      0.94       228



In [18]:
def make_pred(sample): 
    single_pred = classifier.predict([sample])
    return single_pred 

In [19]:
xtrain_candidate_num = round(0.2*len(X_train))
print(xtrain_candidate_num)

182


In [20]:
# ransomware_training_sample_indices = np.where(y_train==1)[0]
# non_zero_mask = X_train[ransomware_training_sample_indices].sum(axis=1) !=0
# valid_indices = ransomware_training_sample_indices[non_zero_mask]

# idx = np.random.choice(valid_indices, 182, replace=False)
# xtrain_poison_candidates = X_train[idx] 
idx_loaded = np.load("C:/Users/Neezy/Videos/attacks_improved_v2/Attacks-Improved_v2/Adversarial_candidates_indices/rf_balanced_df_adversarial_candidate_indices.npy")
xtrain_poison_candidates = X_train[idx_loaded] 
print(len(xtrain_poison_candidates))

182


In [21]:
print(len(xtrain_poison_candidates))

182


In [22]:
perturbation=20
asamples, adv_idx = attack.generate_adversarial_sample(features, idx_loaded, xtrain_poison_candidates, perturbation, classifier)

["Feature: CreateActCtxW changed: {'from': 0, 'to': 4}", "Feature: GlobalMemoryStatusEx changed: {'from': 0, 'to': 7}", "Feature: CreateDirectoryW changed: {'from': 0, 'to': 8}", "Feature: LdrUnloadDll changed: {'from': 0, 'to': 5}", "Feature: SHGetFolderPathW changed: {'from': 0, 'to': 8}", "Feature: SetFileTime changed: {'from': 0, 'to': 12}", "Feature: RegEnumKeyExW changed: {'from': 0, 'to': 3}", "Feature: DrawTextExW changed: {'from': 0, 'to': 2}", "Feature: FindWindowW changed: {'from': 0, 'to': 2}", "Feature: RemoveDirectoryW changed: {'from': 0, 'to': 20}", "Feature: GetKeyboardState changed: {'from': 0, 'to': 5}", "Feature: SendNotifyMessageW changed: {'from': 0, 'to': 16}", "Feature: DrawTextExA changed: {'from': 0, 'to': 5}", "Feature: SHGetSpecialFolderLocation changed: {'from': 0, 'to': 4}", "Feature: gethostbyname changed: {'from': 0, 'to': 17}", "Feature: CertOpenStore changed: {'from': 0, 'to': 2}"]
["Feature: SHGetFolderPathW changed: {'from': 0, 'to': 4}", "Feature: F

In [23]:
np.save("rf_adversarial_samples_bal_functionality_preserving.npy", np.array(asamples))

In [24]:
print(len(asamples))
print(f"AGR: {len(asamples)/len(xtrain_poison_candidates):.4f}") 

166
AGR: 0.9121


In [25]:
### Adversarial Retrain ###  
mask = np.ones(X_train.shape[0], dtype=bool)
mask[adv_idx] = False

X_train_base_samples_removed = X_train[mask]
y_train_base_samples_removed = y_train[mask]
print(f"length of Xtrain before successful base adversarial samples removed: {len(X_train)}")
print(f"length of Xtrain with successful base adversarial samples removed: {len(X_train_base_samples_removed)}")

### Select a hold out test set of the generated adversarial samples
k = round(0.2 * len(asamples)) 
print(f"20% of the adversarial samples held out for testing: {k}")
remove_samples_from_asamples = random.sample(asamples, k)
print(len(remove_samples_from_asamples))

### remove test set from total adversarial samples and add adversarial samples to the training set 
result = [item for item in asamples if item not in remove_samples_from_asamples]
np_result = np.array(result)
print(len(result))
poisoned_xtrain = np.concatenate((X_train_base_samples_removed, np_result))
print(f"Total length of poisoned Xtrain: {len(poisoned_xtrain)}")

### Add the corresponding amount of ytrain adversarial samples. (correctly labelled as ransomware) 
adv_y = []
for i in range(len(np_result)): 
    adv_y.append(1) 
    
print(len(adv_y))

np_advy = np.array(adv_y)
poisoned_ytrain = np.concatenate((y_train_base_samples_removed, np_advy))
print(f"Total length of poisoned Ytrain: {len(poisoned_ytrain)}") 

if len(poisoned_ytrain) == len(poisoned_xtrain): 
    print("Poisoned Xtrain and ytrain are aligned, posioned training can now commence")
else: 
    print("Not aligned, something is not aligned, check k value")

length of Xtrain before successful base adversarial samples removed: 912
length of Xtrain with successful base adversarial samples removed: 746
20% of the adversarial samples held out for testing: 33
33
101
Total length of poisoned Xtrain: 847
101
Total length of poisoned Ytrain: 847
Poisoned Xtrain and ytrain are aligned, posioned training can now commence


In [26]:
poisoned_classifier = RandomForestClassifier(random_state=42)
poisoned_classifier.fit(poisoned_xtrain, poisoned_ytrain)
y_pred_poisoned = poisoned_classifier.predict(X_test)
poisoned_accuracy = accuracy_score(y_test, y_pred_poisoned)
print(f'Accuracy trained on poisoned samples: {poisoned_accuracy * 100:.2f}%')
cm_poisoned = confusion_matrix(y_test, y_pred_poisoned)
print(f"Confusion Matrix trained on poisoned samples:\n {cm_poisoned}") 
cr_poisoned = classification_report(y_test, y_pred_poisoned)
print(f"Classification report trained on poisoned samples:\n {cr_poisoned}")

Accuracy trained on poisoned samples: 94.74%
Confusion Matrix trained on poisoned samples:
 [[126   2]
 [ 10  90]]
Classification report trained on poisoned samples:
               precision    recall  f1-score   support

           0       0.93      0.98      0.95       128
           1       0.98      0.90      0.94       100

    accuracy                           0.95       228
   macro avg       0.95      0.94      0.95       228
weighted avg       0.95      0.95      0.95       228



In [27]:
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = poisoned_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(remove_samples_from_asamples)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [28]:
###Model transfer ###
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [29]:
### Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Decision Tree ####
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X_train, y_train)
y_pred_dt = classifier_dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt) 
print(f"Accuracy of Decision Tree Target Model: {accuracy_score(y_test, y_pred_dt):.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model = XGBClassifier(**params)
model.fit(X_train, y_train)
y_pred_xgb = model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f'Accuracy of the XGboost Model: {accuracy_xgb * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier = SVC(kernel='poly', C=1.0, random_state=42,max_iter=5000)
svm_classifier.fit(X_train, y_train)
y_pred_svm_poly = svm_classifier.predict(X_test)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test, y_pred_svm_poly):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear = SVC(kernel='linear', C=4.0, random_state=42,max_iter=5000)
svm_classifier_linear.fit(X_train, y_train)
y_pred_svm_linear = svm_classifier_linear.predict(X_test)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test, y_pred_svm_linear):.2f}")

### Logistic Regression ###
LR_model = LogisticRegression(solver='saga',max_iter=5000)
LR_model.fit(X_train, y_train)
ypred_lr = LR_model.predict(X_test)
accuracy = accuracy_score(y_test, ypred_lr)
print(f"accuracy of the Logistic Regression Model: {round(accuracy,2)}")

Accuracy of Decision Tree Target Model: 0.94
Accuracy of the XGboost Model: 92.11%
Accuracy of the SVM Classifier (Kernel=Poly): 0.46
Accuracy of the SVM Classifier (Kernel=Linear) : 0.87


c:\Users\Neezy\anaconda3\envs\tf\lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


accuracy of the Logistic Regression Model: 0.83


c:\Users\Neezy\anaconda3\envs\tf\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [30]:
def evasion_attack(adversarial_samples,
                   DT_Model,
                   XGB_Model, 
                   SVM_poly_model, 
                   SVM_Linear,
                   Logistic_reg_model):
    
    predictions = {
        "Decision_Tree":[],
        "XGB":[],
        "SVM_poly": [], 
        "SVM_Linear":[], 
        "Logistic_Regression":[]   
    }
    
    models = {
        "Decision_Tree": DT_Model,
        "XGB":XGB_Model,
        "SVM_poly": SVM_poly_model,
        "SVM_Linear":SVM_Linear, 
        "Logistic_Regression":Logistic_reg_model
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [31]:
res = evasion_attack(asamples, classifier_dt, model, svm_classifier, svm_classifier_linear, LR_model)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = df.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names))      


 Decision_Tree: predictions

 Decision_Tree evasion rate: 146/166 samples evaded

 evaded samples:  Counter({'GandCrab': 15, 'None': 14, 'Azov': 7, 'UpdateRecords.html': 7, 'NominatusStorm': 6, 'UpdateForm.html': 6, 'Babuk': 5, 'Conti': 4, 'Sodinokibi': 4, 'UpdateInformation.html': 3, 'Snatch': 3, 'Attachment.html': 3, 'Hive': 2, 'Chaos': 2, 'Ryuk': 2, 'Egregor': 2, 'Balaclava': 2, 'Nitro': 2, 'RedLineStealer': 2, 'Attached.html': 2, 'Dharma': 2, 'UpdateNow.html': 1, 'InfinityLock': 1, 'Attachment774.html': 1, 'Nefilim': 1, 'eCh0raix': 1, 'LimeRAT': 1, 'BlackOut': 1, 'Win32.SdBot': 1, 'Zeoticus': 1, 'HiddenTear': 1, 'Matrix': 1, 'JigsawLocker': 1, 'AttachedForm858.html': 1, 'DarkWorld': 1, 'PP-Form445.html': 1, 'OrcusRAT': 1, 'AttachedForm699.html': 1, 'LilithCrypt': 1, 'Form350.html': 1, 'UpdateForm447.html': 1, 'ChinaJm': 1, 'Mimikatz': 1, 'EternityStealer': 1, 'RubberDucky': 1, 'PP-Form958.html': 1, 'Targeted': 1, 'ESCAL': 1, 'Mydoom': 1, 'Attachment581.html': 1, 'Jcrypt': 1, 'Doej

In [32]:
scaler = StandardScaler() 

x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

asamples_scaled = scaler.transform(asamples)

In [33]:
### Decision Tree Scaled####
classifier_dt_scaled = DecisionTreeClassifier(random_state=42)
classifier_dt_scaled.fit(x_train_scaled, y_train)
y_pred_dt_scaled = classifier_dt_scaled.predict(x_test_scaled)
accuracy_dt_scaled = accuracy_score(y_test, y_pred_dt_scaled) 
print(f"Accuracy of Decision Tree Target Model: {accuracy_score(y_test, y_pred_dt_scaled):.2f}")

### XGboost Scaled #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model_scaled = XGBClassifier(**params)
model_scaled.fit(x_train_scaled, y_train)
y_pred_xgb_scaled = model.predict(x_test_scaled)
accuracy_xgb_scaled = accuracy_score(y_test, y_pred_xgb_scaled)
print(f'Accuracy of the XGboost Model: {accuracy_xgb_scaled * 100:.2f}%') 

### SVM Classifier Kernel=Poly Scaled ###
svm_classifier_scaled = SVC(kernel='poly', C=1.0, random_state=42, max_iter=5000)
svm_classifier_scaled.fit(x_train_scaled, y_train)
y_pred_svm_poly_scaled = svm_classifier_scaled.predict(x_test_scaled)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test, y_pred_svm_poly_scaled):.2f}")

### SVM Classifier Kernel=Linear Scaled ###
svm_classifier_linear_scaled = SVC(kernel='linear', C=4.0, random_state=42, max_iter=5000)
svm_classifier_linear_scaled.fit(x_train_scaled, y_train)
y_pred_svm_linear_scaled = svm_classifier_linear.predict(x_test_scaled)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test, y_pred_svm_linear_scaled):.2f}")

### Logistic Regression Scaled ###
LR_model_scaled = LogisticRegression(solver='saga',max_iter=5000)
LR_model_scaled.fit(x_train_scaled, y_train)
ypred_lr_scaled = LR_model_scaled.predict(x_test_scaled)
accuracy = accuracy_score(y_test, ypred_lr_scaled)
print(f"accuracy of the Scaled Logistic Regression Model: {round(accuracy,2)}")

Accuracy of Decision Tree Target Model: 0.94
Accuracy of the XGboost Model: 43.86%
Accuracy of the SVM Classifier (Kernel=Poly): 0.57
Accuracy of the SVM Classifier (Kernel=Linear) : 0.44
accuracy of the Scaled Logistic Regression Model: 0.92


In [34]:
#### EVASION FUNCTION SCALED ########
def evasion_attack_scaled(adversarial_samples_scaled,
                   DT_Model_scaled,
                   XGB_Model_scaled, 
                   SVM_poly_model_scaled, 
                   SVM_linear_model_scaled, 
                   Logistic_reg_model_scaled):
    
    predictions = {
        "Decision_Tree_scaled":[],
        "XGB_scaled":[],
        "SVM_poly_scaled": [], 
        "SVM_Linear_scaled":[], 
        "Logistic_Regression_scaled":[]   
    }
    
    models = {
        "Decision_Tree": DT_Model_scaled,
        "XGB":XGB_Model_scaled,
        "SVM_poly": SVM_poly_model_scaled, 
        "SVM_Linear":SVM_linear_model_scaled, 
        "Logistic_Regression":Logistic_reg_model_scaled
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples_scaled)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [35]:
res_scaled = evasion_attack_scaled(asamples_scaled, classifier_dt_scaled, model_scaled, svm_classifier_scaled, svm_classifier_linear_scaled, LR_model_scaled)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res_scaled.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = df.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names)) 


 Decision_Tree_scaled: predictions

 No samples evaded for Decision_Tree_scaled

 Decision_Tree_scaled evasion rate: 0/0 samples evaded

 evaded samples:  Counter()

 XGB_scaled: predictions

 No samples evaded for XGB_scaled

 XGB_scaled evasion rate: 0/0 samples evaded

 evaded samples:  Counter()

 SVM_poly_scaled: predictions

 No samples evaded for SVM_poly_scaled

 SVM_poly_scaled evasion rate: 0/0 samples evaded

 evaded samples:  Counter()

 SVM_Linear_scaled: predictions

 No samples evaded for SVM_Linear_scaled

 SVM_Linear_scaled evasion rate: 0/0 samples evaded

 evaded samples:  Counter()

 Logistic_Regression_scaled: predictions

 No samples evaded for Logistic_Regression_scaled

 Logistic_Regression_scaled evasion rate: 0/0 samples evaded

 evaded samples:  Counter()

 Decision_Tree: predictions

 Decision_Tree evasion rate: 146/166 samples evaded

 evaded samples:  Counter({'GandCrab': 15, 'None': 14, 'Azov': 7, 'UpdateRecords.html': 7, 'NominatusStorm': 6, 'UpdateForm